# Fine-tuning BART-base

Switch from `google/flan-t5-small` to `facebook/bart-base` because the FLAN-T5 SentencePiece vocabulary cannot represent JSON braces (`{`, `}`) or several Unicode units (`±`, `µ`, `×`, `≤`, `≥`). BART's byte-level BPE round-trips these characters without `<unk>`.

Reference: [facebook/bart-base model card](https://huggingface.co/facebook/bart-base).


## 1. Install dependencies

In [12]:
%pip install -q "transformers>=4.46,<5" "torch>=2.6" datasets accelerate

## 2. Set up the runtime


In [13]:
import platform
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed

MODEL_NAME = "facebook/bart-base"
SEED = 42
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 4.57.6
Device: cuda
GPU: Tesla T4


## 3. Load the untuned model



In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    use_safetensors=True,
)
model = model.to(device)
model.eval()

assert model.config.is_encoder_decoder
assert not model.training
print("Loaded:", MODEL_NAME)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Model device:", next(model.parameters()).device)
print("Tokenizer limit:", tokenizer.model_max_length)
print("Decoder start token id:", model.config.decoder_start_token_id)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded: facebook/bart-base
Parameters: 139,420,416
Model device: cuda:0
Tokenizer limit: 1000000000000000019884624838656
Decoder start token id: 2


## 4. Load and Prepare dataset
Data is split by device family under `data/splits/`.
Just have to convert each record to input-target pair:
* Input: the `passage` text

* Target: requirements list serialised as a string

In [15]:
!git clone https://github.com/vet3whale/nlp_project.git
%cd nlp_project

fatal: destination path 'nlp_project' already exists and is not an empty directory.
/content/nlp_project


load the three splits

In [16]:
import json, os
import pandas as pd

SPLIT_DIR = "data/splits"

def load_split(name):
    for ext in (".json", ".jsonl"):
        path = os.path.join(SPLIT_DIR, f"{name}{ext}")
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                if ext == ".json":
                    return json.load(f)
                return [json.loads(line) for line in f if line.strip()]
    raise FileNotFoundError(f"No split found for {name} in {SPLIT_DIR}")

train_records = load_split("train")
val_records   = load_split("validation")
test_records  = load_split("test")

verify data before building

In [17]:
# The splits do not carry a 'family' key. Recover the family from the
# per-device files in data/extracted/, joining on passage text.
import glob

family_by_passage = {}
for path in glob.glob("data/extracted/*.json"):
    fam = os.path.splitext(os.path.basename(path))[0]
    with open(path, encoding="utf-8") as f:
        for rec in json.load(f):
            family_by_passage[rec["passage"].strip()] = fam

def family_of(r):
    return family_by_passage.get(r["passage"].strip(), "UNKNOWN")

for name, recs in [("train", train_records), ("val", val_records), ("test", test_records)]:
    fams = sorted({family_of(r) for r in recs})
    negs = sum(1 for r in recs if not (r.get("requirements") or []))
    print(f"{name}: {len(recs)} examples | families={fams} | {negs} negatives")

train_f = {family_of(r) for r in train_records}
val_f   = {family_of(r) for r in val_records}
test_f  = {family_of(r) for r in test_records}
print("overlaps:", train_f & val_f, train_f & test_f, val_f & test_f)
assert not (train_f & val_f) and not (train_f & test_f) and not (val_f & test_f)

train: 135 examples | families=['hdc1080', 'ina219', 'opt3001', 'sgp30', 'tmag5273', 'tmp117', 'veml7700'] | 26 negatives
val: 40 examples | families=['mcp9808', 'mpl3115a2'] | 8 negatives
test: 45 examples | families=['bme280', 'fdc1004', 'sht31'] | 9 negatives
overlaps: set() set() set()


build input/target pairs

In [18]:
INSTRUCTION = ("Extract numerical operating requirements and their conditions "
               "from the datasheet passage. Return JSON.")

def serialize_requirements(reqs):
    return json.dumps(reqs or [], ensure_ascii=False, separators=(",", ":"))

def build_frame(records):
    rows = []
    for r in records:
        reqs = r.get("requirements") or []
        rows.append({
            "input": f"{INSTRUCTION}\n\n{r['passage'].strip()}",
            "target": serialize_requirements(reqs),
            "family": family_of(r),
            "n_reqs": len(reqs),
        })
    return pd.DataFrame(rows)

df_train = build_frame(train_records)
df_val   = build_frame(val_records)
df_test  = build_frame(test_records) # built for later; not used in training

## 5. Tokenise inputs and targets
BART uses byte-level BPE, so all ASCII punctuation and Unicode units round-trip. Measure lengths first, then choose a target cap from the observed maximum.


In [19]:
import numpy as np

def raw_lengths(texts, is_target=False):
    kw = {"text_target": texts} if is_target else {"text": texts}
    enc = tokenizer(**kw, truncation=False, padding=False, add_special_tokens=True)
    return [len(ids) for ids in enc["input_ids"]]

for name, df in [("train", df_train), ("val", df_val)]:
    li = raw_lengths(df["input"].tolist())
    lt = raw_lengths(df["target"].tolist(), is_target=True)
    print(f"{name} input  tokens: p50={np.percentile(li,50):.0f} p95={np.percentile(li,95):.0f} max={max(li)}")
    print(f"{name} target tokens: p50={np.percentile(lt,50):.0f} p95={np.percentile(lt,95):.0f} max={max(lt)}")

train input  tokens: p50=70 p95=112 max=151
train target tokens: p50=70 p95=153 max=257
val input  tokens: p50=72 p95=90 max=118
val target tokens: p50=60 p95=116 max=137


tokenise with correct cap

In [20]:
MAX_INPUT_LEN  = 512
MAX_TARGET_LEN = 384

def tokenize(batch):
    enc = tokenizer(
        batch["input"],
        max_length=MAX_INPUT_LEN, truncation=True, padding=False,
        add_special_tokens=True,
    )
    lab = tokenizer(
        text_target=batch["target"],
        max_length=MAX_TARGET_LEN, truncation=True, padding=False,
    )
    enc["labels"] = lab["input_ids"]
    return enc

In [21]:
def decode_labels(ids):
    return tokenizer.decode([t for t in ids if t != -100], skip_special_tokens=True)

bad = []
for df, name in [(df_train, "train"), (df_val, "val")]:
    for i, row in df.iterrows():
        enc = tokenizer(text_target=row["target"], truncation=False, add_special_tokens=True)
        dec = decode_labels(enc["input_ids"])
        if dec != row["target"]:
            bad.append((name, i))
print("round-trip mismatches:", len(bad))
assert len(bad) == 0, f"{len(bad)} mismatches — do not train yet"

round-trip mismatches: 0


check unicode characters are correctly interpreted

In [22]:
for ch in ['{', '}', '<', '>', '<=', '>=', '°', '±', 'µ', '×', '≤', '≥', '—', ';']:
    ids = tokenizer(ch, add_special_tokens=False)["input_ids"]
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"{ch!r:6} -> {toks}  unk={'<unk>' in toks}")


'{'    -> ['{']  unk=False
'}'    -> ['}']  unk=False
'<'    -> ['<']  unk=False
'>'    -> ['>']  unk=False
'<='   -> ['<', '=']  unk=False
'>='   -> ['>', '=']  unk=False
'°'    -> ['Â°']  unk=False
'±'    -> ['Â±']  unk=False
'µ'    -> ['Â', 'µ']  unk=False
'×'    -> ['ÃĹ']  unk=False
'≤'    -> ['âī', '¤']  unk=False
'≥'    -> ['âī', '¥']  unk=False
'—'    -> ['âĢĶ']  unk=False
';'    -> [';']  unk=False


In [23]:
EOS_ID = tokenizer.eos_token_id
lab_ids = tokenizer(text_target=df_train["target"].tolist(),
                    truncation=False, padding=False)["input_ids"]
print("eos:", EOS_ID, repr(tokenizer.eos_token))
print("all labels end with EOS:", all(s[-1] == EOS_ID for s in lab_ids))
print("labels over cap:", sum(1 for s in lab_ids if len(s) > MAX_TARGET_LEN))
assert all(s[-1] == EOS_ID for s in lab_ids)
assert sum(1 for s in lab_ids if len(s) > MAX_TARGET_LEN) == 0

eos: 2 '</s>'
all labels end with EOS: True
labels over cap: 0


In [24]:
from datasets import Dataset

keep = ["input", "target"]
ds_train = Dataset.from_pandas(df_train[keep], preserve_index=False)
ds_val   = Dataset.from_pandas(df_val[keep],   preserve_index=False)
ds_test  = Dataset.from_pandas(df_test[keep],  preserve_index=False)

tok_train = ds_train.map(tokenize, batched=True, remove_columns=ds_train.column_names)
tok_val   = ds_val.map(tokenize,   batched=True, remove_columns=ds_val.column_names)
tok_test  = ds_test.map(tokenize,  batched=True, remove_columns=ds_test.column_names)

for n, d in [("train", tok_train), ("val", tok_val), ("test", tok_test)]:
    print(n, len(d), "| cols:", d.column_names,
          "| max label len:", max(len(x) for x in d["labels"]))

Map:   0%|          | 0/135 [00:00<?, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Map:   0%|          | 0/45 [00:00<?, ? examples/s]

train 135 | cols: ['input_ids', 'attention_mask', 'labels'] | max label len: 257
val 40 | cols: ['input_ids', 'attention_mask', 'labels'] | max label len: 137
test 45 | cols: ['input_ids', 'attention_mask', 'labels'] | max label len: 113


## 6. Data collator
Collator sits between tokenised dataset and model's forward pass.
Does 2 things:
1. Pads each batch to longest sequence.  
    The collator pads each batch to the longest member of that batch, not to the global `MAX_INPUT_LEN/MAX_TARGET_LEN`. Short batches stay cheap.
2. Masks label padding with -100.  
    PyTorch cross-entropy uses ignore_index=-100, so padding positions in labels are excluded from loss. Without this, the model would learn to predict padding tokens.

In [25]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

sample_batch = data_collator([tok_train[i] for i in range(4)])
print({k: tuple(v.shape) for k, v in sample_batch.items()})
print("padding token in labels:", (sample_batch["labels"] == -100).any().item())

{'input_ids': (4, 79), 'attention_mask': (4, 79), 'labels': (4, 77), 'decoder_input_ids': (4, 77)}
padding token in labels: True


## 7. Training arguments

In [22]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-base-datasheet",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

## 8. Trainer and training

In [23]:
# build trainer
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=data_collator,
)

In [24]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.853000,0.426200
2,0.588900,0.208112
3,0.218700,0.170789
4,0.198300,0.153175
5,0.103500,0.158571
6,0.101900,0.142072
7,0.060100,0.144445
8,0.072000,0.142914
9,0.056500,0.158257
10,0.053100,0.144902


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=340, training_loss=0.1869828313589096, metrics={'train_runtime': 568.5634, 'train_samples_per_second': 4.749, 'train_steps_per_second': 0.598, 'total_flos': 181138172866560.0, 'train_loss': 0.1869828313589096, 'epoch': 20.0})

## 9. Save the fine-tuned model

In [25]:
from google.colab import drive
import os
import json

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the save path in your Drive
SAVE_DIR = "/content/drive/MyDrive/bart-base-datasheet-finetuned"
os.makedirs(SAVE_DIR, exist_ok=True)

# 3. Save model and tokenizer to Drive
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# 4. Save config to Drive
config = {
    "model": MODEL_NAME,
    "instruction": INSTRUCTION,
    "max_input_len": MAX_INPUT_LEN,
    "max_target_len": MAX_TARGET_LEN,
    "seed": SEED,
}

config_path = os.path.join(SAVE_DIR, "training_config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"Saved model, tokenizer, and config to: {SAVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved model, tokenizer, and config to: /content/drive/MyDrive/bart-base-datasheet-finetuned


10. Evaluate Model

In [26]:
import json, numpy as np

def parse(s):
    try: return json.loads(s)
    except: return None

def normalize_reqs(reqs):
    """Normalizes formatting so exact match isn't ruined by key order, int vs float, etc."""
    if not isinstance(reqs, list): return []
    normalized = []
    for r in reqs:
        if not isinstance(r, dict): continue
        norm_r = {
            "parameter": str(r.get("parameter", "")).strip().lower(),
            "operator": str(r.get("operator", "")).strip(),
            # Normalize numbers to float, fallback to string
            "value": float(r.get("value", 0)) if str(r.get("value", "")).replace('.','',1).replace('-','',1).isdigit() else str(r.get("value", "")),
            # Normalize units (e.g., µ -> u, strip °)
            "unit": str(r.get("unit", "")).strip().lower().replace("µ", "u").replace("°", ""),
            "condition": str(r.get("condition", "")).strip().lower()
        }
        normalized.append(norm_r)
    # Sort array so order doesn't matter
    return sorted(normalized, key=lambda x: (x["parameter"], str(x["value"])))

def safe_decode(p):
    # Handle tuple output (transformers 4.57 returns tuple for beam search)
    if isinstance(p, tuple): p = p[0]
    arr = np.asarray(p)
    arr = np.where(arr < 0, tokenizer.pad_token_id, arr)
    return tokenizer.batch_decode(arr, skip_special_tokens=True)

model.eval()
preds = trainer.predict(
    tok_val,
    max_new_tokens=MAX_TARGET_LEN,
    num_beams=8,
    length_penalty=2.0,  # Change to LOCKED_LP if you're running the sweep
)

decoded_preds  = safe_decode(preds.predictions)
gold_targets   = df_val["target"].tolist()

# --- Extraction Accuracy Metrics (Proposal Method) ---
pos_correct = pos_total = 0
neg_correct = neg_total = 0

for p_str, g_str in zip(decoded_preds, gold_targets):
    p = parse(p_str)
    g = parse(g_str)

    p_norm = normalize_reqs(p) if p is not None else []
    g_norm = normalize_reqs(g)

    if len(g_norm) == 0:
        neg_total += 1
        if len(p_norm) == 0:
            neg_correct += 1
    else:
        pos_total += 1
        if p_norm == g_norm:
            pos_correct += 1

pos_acc = pos_correct / pos_total if pos_total else 0
neg_acc = neg_correct / neg_total if neg_total else 0

print(f"Positive Extraction Accuracy : {pos_correct}/{pos_total} ({pos_acc:.3f})")
print(f"Negative Extraction Accuracy : {neg_correct}/{neg_total} ({neg_acc:.3f})")
print(f"val loss                     : {preds.metrics.get('test_loss', 'n/a')}")

# --- Side-by-side Mismatches ---
# Find true mismatches using the normalized comparison
mismatch_idx = [
    i for i in range(len(decoded_preds))
    if normalize_reqs(parse(decoded_preds[i])) != normalize_reqs(parse(gold_targets[i]))
]

print(f"\n=== showing up to 5 mismatches ===")
for i in mismatch_idx[:5]:
    print(f"\n--- val[{i}] ---")
    print("PASSAGE:", df_val["input"].iloc[i][-250:].replace("\n", " ")[:250])
    print("PRED  :", decoded_preds[i][:400])
    print("GOLD  :", gold_targets[i][:400])

# --- Side-by-side Negatives ---
neg_idx = [i for i, g in enumerate(gold_targets) if len(normalize_reqs(parse(g))) == 0][:2]
print(f"\n=== showing 2 negatives ===")
for i in neg_idx:
    print(f"\n--- val[{i}] (negative) ---")
    print("PASSAGE:", df_val["input"].iloc[i][-250:].replace("\n", " ")[:250])
    print("PRED  :", decoded_preds[i][:200])
    print("GOLD  :", gold_targets[i][:200])

Positive Extraction Accuracy : 9/32 (0.281)
Negative Extraction Accuracy : 2/8 (0.250)
val loss                     : 0.14207178354263306

=== showing up to 5 mismatches ===

--- val[1] ---
PASSAGE: quirements and their conditions from the datasheet passage. Return JSON.  9.2 Absolute maximum ratings, Table 4 ESD and latchup protection characteristics. Human body model: ±2000 V. Charge device model: ±500 V. Latchup current at T = 85 °C: ±100 mA.
PRED  : [{"parameter":"ESD and latchup protection characteristics","operator":"<=","value":2000,"unit":"mA","condition":"Absolute maximum ratings, Table 4; figure contents not included in this passage"}]
GOLD  : []

--- val[2] ---
PASSAGE: Extract numerical operating requirements and their conditions from the datasheet passage. Return JSON.  9.3 Mechanical characteristics VDD = 2.5 V; T = 25 °C; Calibrated range Pressure measurement range MIN: 50 | MAX: 110 | UNIT: kPa
PRED  : [{"parameter":"Pressure measurement range","operator":">=","value":5

## 11. Hyperparameter sweep

Stage 10 established the baseline: exact match **0.325**, requirement-level precision **1.000**, recall **0.447**, F1 **0.618**, negatives **4/8** at `length_penalty=2.0`, `num_beams=8`.

The following stage tunes three hyperparameters - **learning rate**, **num_train_epochs**, **per_device_train_batch_size**

### Sweep strategy

Sequential single-dimension sweep, not a full grid. On 40 val examples a grid overfits to val; sequential sweeps give a defensible answer in 4-5 runs.

- **Round 1:** `learning_rate ∈ {2e-5, 3e-5, 5e-5}`, everything else fixed at the Stage 10 baseline.
- **Round 2:** `num_train_epochs ∈ {10, 15, 20}` at the winning LR.
- **Round 3:** `per_device_train_batch_size ∈ {4, 8}` at winning LR + epochs

### Frozen during the sweep

`INSTRUCTION`, `tok_train`, `tok_val`, `MAX_INPUT_LEN`, `MAX_TARGET_LEN`, `SEED`, `data_collator`, `weight_decay`, `eval_strategy`, `save_strategy`.

In [9]:
import gc
import json
import math
import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed,
)

# Release the baseline model and its optimizer before loading another model.
for name in ("trainer", "model", "data_collator", "sample_batch", "preds"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def parse_requirements(text):
    """Normalize valid requirements; return None for invalid output, never []."""
    try:
        reqs = json.loads(text)
        if not isinstance(reqs, list):
            return None
        normalized = []
        for r in reqs:
            if not isinstance(r, dict) or set(r) != {"parameter", "operator", "value", "unit", "condition"}:
                return None
            if not all(isinstance(r[k], str) for k in ("parameter", "operator", "unit")):
                return None
            if r["condition"] is not None and not isinstance(r["condition"], str):
                return None
            value = float(r["value"])
            if isinstance(r["value"], bool) or not math.isfinite(value):
                return None
            normalized.append((
                r["parameter"].strip().lower(), r["operator"].strip(), value,
                r["unit"].strip().replace("µ", "u").replace("μ", "u"),
                "" if r["condition"] is None else r["condition"].strip().lower(),
            ))
        return sorted(normalized)
    except (ValueError, TypeError):
        return None


def score_predictions(decoded, gold_texts):
    predicted = [parse_requirements(s) for s in decoded]
    gold = [parse_requirements(s) for s in gold_texts]
    assert len(predicted) == len(gold) and len(gold) > 0
    assert all(g is not None for g in gold), "Invalid gold annotation"
    positive = np.array([len(g) > 0 for g in gold])
    correct = np.array([p is not None and p == g for p, g in zip(predicted, gold)])
    return {
        "accuracy": float(correct.mean()),
        "pos_acc": float(correct[positive].mean()) if positive.any() else float("nan"),
        "neg_acc": float(correct[~positive].mean()) if (~positive).any() else float("nan"),
        "invalid_outputs": sum(p is None for p in predicted),
    }


In [28]:
# Keep sweep settings here so they do not depend on an earlier runtime cell.
SEED = 42
MODEL_NAME = "facebook/bart-base"
MAX_TARGET_LEN = 384


def run_config(lr, epochs):
    set_seed(SEED)
    run_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, use_safetensors=True)
    run_trainer = None
    try:
        args = Seq2SeqTrainingArguments(
            output_dir=f"./sweep_runs/lr{lr}_ep{epochs}",
            learning_rate=lr,
            num_train_epochs=epochs,
            per_device_train_batch_size=8,
            per_device_eval_batch_size=4,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            save_only_model=True,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            predict_with_generate=True,  # Return token IDs, not vocabulary logits.
            eval_accumulation_steps=1,
            fp16=torch.cuda.is_available(),
            seed=SEED,
            logging_steps=10,
            report_to="none",
        )
        run_trainer = Seq2SeqTrainer(
            model=run_model,
            args=args,
            train_dataset=tok_train,
            eval_dataset=tok_val,
            processing_class=tokenizer,
            data_collator=DataCollatorForSeq2Seq(tokenizer, model=run_model),
        )
        run_trainer.train()
        prediction = run_trainer.predict(
            tok_val,
            max_new_tokens=MAX_TARGET_LEN,
            num_beams=8,
            length_penalty=1.0,
            no_repeat_ngram_size=0,  # JSON keys must repeat across requirements.
            do_sample=False,
        )
        ids = np.where(prediction.predictions < 0, tokenizer.pad_token_id, prediction.predictions)
        decoded = tokenizer.batch_decode(ids, skip_special_tokens=True)
        scores = score_predictions(decoded, df_val["target"].tolist())
        return {
            "lr": lr, "epochs": epochs, **scores,
            "val_loss": float(prediction.metrics["test_loss"]),
            "checkpoint": run_trainer.state.best_model_checkpoint,
        }
    finally:
        del run_trainer, run_model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_sweep(configs):
    required = ("tokenizer", "tok_train", "tok_val", "df_val", "score_predictions", "set_seed")
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            "Missing sweep prerequisites: " + ", ".join(missing)
            + ". Run Sections 1-6 and the sweep setup cell in this kernel first. "
            "You do not need to rerun baseline training (Sections 7-10)."
        )
    results = []
    for lr, epochs in configs:
        print(f"\nTraining lr={lr:g}, epochs={epochs}")
        result = run_config(lr, epochs)
        results.append(result)
        pd.DataFrame(results).to_csv("sweep_results.csv", index=False)
        print(f"Positive: {result['pos_acc']:.1%} | Negative: {result['neg_acc']:.1%} | "
              f"Overall: {result['accuracy']:.1%} | Loss: {result['val_loss']:.4f}")
    return pd.DataFrame(results).sort_values(
        ["pos_acc", "val_loss"], ascending=[False, True]
    ).reset_index(drop=True)


run_sweep([(5e-5, 5), (5e-5, 10), (5e-5, 20)])



Training lr=5e-05, epochs=5


Epoch,Training Loss,Validation Loss
1,2.636900,0.380264
2,0.455200,0.252933
3,0.179400,0.231394
4,0.172400,0.213068
5,0.091100,0.202713


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 25.0% | Negative: 0.0% | Overall: 20.0% | Loss: 0.2027

Training lr=5e-05, epochs=10


Epoch,Training Loss,Validation Loss
1,2.630000,0.367531
2,0.431200,0.254474
3,0.169700,0.246745
4,0.162600,0.190267
5,0.093700,0.185919
6,0.077400,0.191857
7,0.040800,0.199501
8,0.053700,0.182583
9,0.041200,0.209680
10,0.046300,0.199674


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


... (3 outputs left out)

Epoch,Training Loss,Validation Loss
1,2.630300,0.363581
2,0.428200,0.252760
3,0.162300,0.244220
4,0.156400,0.211314
5,0.088600,0.207662
6,0.076300,0.203846
7,0.048900,0.193153
8,0.050200,0.202720
9,0.040300,0.191810
10,0.036100,0.189370


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 50.0% | Negative: 62.5% | Overall: 52.5% | Loss: 0.1852


,lr,epochs,accuracy,pos_acc,neg_acc,invalid_outputs,val_loss,checkpoint
0,0.00005,20,0.525,0.50000,0.625,8,0.185198,./sweep_runs/lr5e-05_ep20/checkpoint-323
1,0.00005,10,0.325,0.28125,0.500,4,0.182583,./sweep_runs/lr5e-05_ep10/checkpoint-136
2,0.00005,5,0.200,0.25000,0.000,16,0.202713,./sweep_runs/lr5e-05_ep5/checkpoint-85


In [29]:
run_sweep([(1e-4, 20)])


Training lr=0.0001, epochs=20


Epoch,Training Loss,Validation Loss
1,3.092300,0.341551
2,0.508700,0.212663
3,0.184000,0.190783
4,0.129600,0.258420
5,0.078800,0.220306
6,0.065800,0.216444
7,0.056600,0.191711
8,0.052100,0.198818
9,0.030000,0.237341
10,0.053900,0.237103


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 21.9% | Negative: 37.5% | Overall: 25.0% | Loss: 0.1908


,lr,epochs,accuracy,pos_acc,neg_acc,invalid_outputs,val_loss,checkpoint
0,0.0001,20,0.25,0.21875,0.375,9,0.190783,./sweep_runs/lr0.0001_ep20/checkpoint-51


In [31]:
run_sweep([(6e-5, 20), (7e-5, 20), (8e-5, 20), (9e-5, 20)])


Training lr=6e-05, epochs=20


Epoch,Training Loss,Validation Loss
1,2.611600,0.341425
2,0.442000,0.245564
3,0.164700,0.258622
4,0.155100,0.221783
5,0.071000,0.216847
6,0.057900,0.208202
7,0.040600,0.216438
8,0.054900,0.217284
9,0.032500,0.220569
10,0.036500,0.200133


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 50.0% | Negative: 50.0% | Overall: 50.0% | Loss: 0.1991

Training lr=7e-05, epochs=20


Epoch,Training Loss,Validation Loss
1,2.615900,0.316139
2,0.460800,0.242086
3,0.166800,0.202406
4,0.136400,0.224254
5,0.080300,0.192909
6,0.056400,0.218034
7,0.035800,0.172221
8,0.055000,0.188323
9,0.032200,0.189840
10,0.040800,0.182174


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


... (4 outputs left out)

Epoch,Training Loss,Validation Loss
1,2.709200,0.312142
2,0.482100,0.196497
3,0.165100,0.201672
4,0.138600,0.279060
5,0.074600,0.234896
6,0.064300,0.232477
7,0.047900,0.181869
8,0.050500,0.170322
9,0.027700,0.189311
10,0.032300,0.169252


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 59.4% | Negative: 87.5% | Overall: 65.0% | Loss: 0.1587


,lr,epochs,accuracy,pos_acc,neg_acc,invalid_outputs,val_loss,checkpoint
0,0.00009,20,0.650,0.59375,0.875,5,0.158661,./sweep_runs/lr9e-05_ep20/checkpoint-238
1,0.00007,20,0.625,0.53125,1.000,4,0.172221,./sweep_runs/lr7e-05_ep20/checkpoint-119
2,0.00006,20,0.500,0.50000,0.500,6,0.199144,./sweep_runs/lr6e-05_ep20/checkpoint-187
3,0.00008,20,0.425,0.40625,0.500,6,0.183297,./sweep_runs/lr8e-05_ep20/checkpoint-51


In [32]:
run_sweep([(9e-5, 25), (9e-5, 30)])


Training lr=9e-05, epochs=25


Epoch,Training Loss,Validation Loss
1,2.709300,0.312569
2,0.482900,0.209996
3,0.171200,0.229444
4,0.144000,0.239294
5,0.083000,0.183886
6,0.082700,0.174433
7,0.053000,0.201096
8,0.044100,0.196614
9,0.028100,0.207089
10,0.029400,0.213272


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 40.6% | Negative: 75.0% | Overall: 47.5% | Loss: 0.1744

Training lr=9e-05, epochs=30


Epoch,Training Loss,Validation Loss
1,2.709600,0.312611
2,0.487900,0.200142
3,0.160100,0.188015
4,0.134500,0.263265
5,0.069900,0.163580
6,0.068100,0.191836
7,0.047300,0.210481
8,0.039300,0.208984
9,0.031900,0.214644
10,0.043000,0.226535


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Positive: 31.2% | Negative: 50.0% | Overall: 35.0% | Loss: 0.1636


,lr,epochs,accuracy,pos_acc,neg_acc,invalid_outputs,val_loss,checkpoint
0,0.00009,25,0.475,0.40625,0.75,12,0.174433,./sweep_runs/lr9e-05_ep25/checkpoint-102
1,0.00009,30,0.350,0.31250,0.50,3,0.163580,./sweep_runs/lr9e-05_ep30/checkpoint-85
